# 2D-to-3D — Hunyuan3D-2mini on Colab T4

**Run All** twice: 1st installs + restarts, 2nd generates your 3D model.

Runtime → **T4 GPU** | ~1.2GB model | Loads in seconds | Generates in 1-3 min

In [ ]:
#@title 1. Install (~3 min)
import os, sys, subprocess
REPO = '/content/2d-to-3d-game-models'
HY3D = '/content/Hunyuan3D-2.1'
M = '/content/.mini_ok2'
if not os.path.exists(M):
    os.chdir('/content')
    subprocess.run(['rm','-rf',REPO])
    subprocess.check_call(['git','clone','-b','claude/image-to-3d-pipeline-CnSII',
        'https://github.com/pmikola/2d-to-3d-game-models.git'])
    if not os.path.exists(HY3D):
        subprocess.check_call(['git','clone',
            'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git',HY3D])
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
        'scipy','onnxruntime-gpu','onnxruntime'])
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
        'transformers','diffusers','accelerate','safetensors',
        'huggingface_hub','einops','omegaconf','pyyaml',
        'trimesh','pygltflib','xatlas','Pillow',
        'opencv-python','imageio','scikit-image',
        'tqdm','ninja','pybind11','timm'])
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
        '--no-deps','rembg==2.0.57'])
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
        'pooch','pymatting','filetype','imagehash'])

    # Stub out pymeshlab so Hunyuan3D imports don't crash
    # (pymeshlab fails to compile on Colab but hy3dshape imports it)
    stub = os.path.join(HY3D, 'hy3dshape', 'hy3dshape', 'pymeshlab_stub.py')
    with open(stub, 'w') as f:
        f.write('# Stub module — pymeshlab cannot compile on Colab\n')
        f.write('class MeshSet:\n')
        f.write('    def __init__(self): pass\n')
        f.write('    def __getattr__(self, name): return lambda *a,**k: None\n')
        f.write('class Mesh:\n')
        f.write('    def __init__(self, *a, **k): pass\n')

    # Create pymeshlab as importable package pointing to stub
    import site
    sp = site.getsitepackages()[0]
    pymeshlab_init = os.path.join(sp, 'pymeshlab', '__init__.py')
    os.makedirs(os.path.dirname(pymeshlab_init), exist_ok=True)
    with open(pymeshlab_init, 'w') as f:
        f.write('# Stub — pymeshlab not available on Colab\n')
        f.write('class MeshSet:\n')
        f.write('    def __init__(self): pass\n')
        f.write('    def __getattr__(self, name): return lambda *a,**k: None\n')
        f.write('class Mesh:\n')
        f.write('    def __init__(self, *a, **k): pass\n')
        f.write('def Percentage(v): return v\n')

    print('All installed (pymeshlab stubbed).')
    open(M,'w').write('ok')
    print('Restarting...')
    try:
        import IPython; IPython.get_ipython().kernel.do_shutdown(True)
    except: os._exit(0)
else:
    os.chdir(REPO)
    import torch
    if torch.cuda.is_available():
        v=torch.cuda.get_device_properties(0).total_memory/1024**3
        print(f'GPU: {torch.cuda.get_device_name(0)} ({v:.0f}GB) | Ready!')
    import psutil; r=psutil.virtual_memory()
    print(f'RAM: {r.available/1024**3:.1f}GB free / {r.total/1024**3:.0f}GB')

In [ ]:
#@title 2. Upload image
from google.colab import files
from PIL import Image
from IPython.display import display
import numpy as np
P='/content/input.png'
print('Upload PNG/JPG:')
u=files.upload()
if u:
    import shutil; shutil.copy(list(u.keys())[0],P)
    img=Image.open(P); print(f'{img.size}'); display(img.resize((300,300)))
else:
    a=np.full((512,512,3),220,dtype=np.uint8)
    y,x=np.ogrid[-256:256,-256:256]; a[x**2+y**2<150**2]=[180,80,40]
    Image.fromarray(a).save(P); print('Using test image')

In [ ]:
#@title 3. Generate 3D model
import os, sys, time, shutil, tempfile, base64, gc
import torch, trimesh, numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML
from huggingface_hub import snapshot_download

HY3D='/content/Hunyuan3D-2.1'
REPO='/content/2d-to-3d-game-models'
P='/content/input.png'
OUT='/content/output/model.glb'
RAW='/content/output/raw.glb'
os.makedirs('/content/output',exist_ok=True)

sys.path.insert(0,HY3D)
sys.path.insert(0,os.path.join(HY3D,'hy3dshape'))
os.chdir(HY3D)
t0=time.time()

# --- Download mini weights manually to local dir ---
print('[1/6] Downloading Hunyuan3D-2mini weights...')
local_model = '/content/hy3d_mini_weights'
if not os.path.exists(os.path.join(local_model, 'hunyuan3d-dit-v2-mini')):
    snapshot_download(
        repo_id='tencent/Hunyuan3D-2mini',
        local_dir=local_model,
        allow_patterns=['hunyuan3d-dit-v2-mini/*', 'hunyuan3d-vae-v2-1/*'],
    )
    print(f'  Downloaded to {local_model}')
else:
    print(f'  Cached at {local_model}')

# List what we got
for root, dirs, files in os.walk(local_model):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp) / 1024**2
        if sz > 1:
            print(f'  {os.path.relpath(fp, local_model)}: {sz:.0f}MB')

# --- Load from local path ---
print('\n  Loading pipeline from local weights...')
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    local_model,
    subfolder='hunyuan3d-dit-v2-mini',
    use_safetensors=False,
    torch_dtype=torch.float16,
).to('cuda')
print(f'  Loaded in {time.time()-t0:.0f}s | GPU: {torch.cuda.memory_allocated()/1024**3:.1f}GB')

# --- Background removal ---
print('\n[2/6] Background removal...')
from hy3dshape.rembg import BackgroundRemover
rb=BackgroundRemover()
img=rb(Image.open(P).convert('RGBA'))
del rb; gc.collect()
display(img.resize((200,200)))

# --- Generate ---
print('\n[3/6] Generating 3D (~1-3 min)...')
with torch.no_grad():
    mesh=pipe(image=img, num_inference_steps=50)[0]
mesh.export(RAW)
print(f'  {len(mesh.vertices)} verts, {len(mesh.faces)} faces')
del pipe; gc.collect(); torch.cuda.empty_cache()

# --- Repair ---
os.chdir(REPO); sys.path.insert(0,REPO)
print('\n[4/6] Mesh repair...')
from pipeline.mesh_repair import repair_and_prepare
from pipeline.geometry import normalize_mesh, unwrap_uvs, save_mesh_as_obj
mesh=trimesh.load(RAW,force='mesh')
mesh=repair_and_prepare(mesh, smooth_iterations=3)
normalize_mesh(mesh)
print(f'  {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

# --- UV + PBR ---
print('\n[5/6] UV + PBR maps...')
unwrap_uvs(mesh)
from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps
from pipeline.export import export_textured_dir_to_glb, validate_glb
tex=Image.open(P).convert('RGB').resize((1024,1024))
with tempfile.TemporaryDirectory() as tmp:
    op=save_mesh_as_obj(mesh,tmp)
    td=Path(tmp)/'textured'; td.mkdir()
    tex.save(str(td/'texture_atlas.png'))
    shutil.copy(op,str(td/'mesh_textured.obj'))
    pbr=generate_pbr_maps(tex,strength=1.5)
    save_pbr_maps(pbr,str(td/'pbr'))
    r=Image.new('RGB',(256*4,256))
    r.paste(tex.resize((256,256)),(0,0))
    r.paste(pbr['normal'].resize((256,256)),(256,0))
    r.paste(pbr['roughness'].convert('RGB').resize((256,256)),(512,0))
    r.paste(pbr['metallic'].convert('RGB').resize((256,256)),(768,0))
    display(r)
    print('\n[6/6] Export GLB...')
    export_textured_dir_to_glb(str(td),OUT)

info=validate_glb(OUT)
el=time.time()-t0
sz=os.path.getsize(OUT)/(1024*1024)
print(f'\n{"="*50}')
print(f'DONE in {el:.0f}s | {sz:.1f}MB | {info.get("total_vertices","?")} verts')
print(f'{"="*50}')

with open(OUT,'rb') as f: b=base64.b64encode(f.read()).decode()
display(HTML(f'<h2><a href="data:model/gltf-binary;base64,{b}" download="model.glb" '
    f'style="background:#4CAF50;color:white;padding:15px 30px;'
    f'text-decoration:none;border-radius:8px;font-size:18px;">'
    f'📥 DOWNLOAD model.glb ({sz:.1f}MB)</a></h2>'))
with open(RAW,'rb') as f: br=base64.b64encode(f.read()).decode()
display(HTML(f'<a href="data:model/gltf-binary;base64,{br}" download="raw.glb" '
    f'style="color:#2196F3">Download raw shape</a>'))
print('View: https://gltf-viewer.donmccurdy.com/')